# kombu smoke test

Exercises a representative slice of the kombu public API. Kombu is the messaging library that backs Celery — its primary surfaces are `Connection`, `Exchange`, `Queue`, `Producer`, `Consumer`, and `Message`. We exercise these in-memory (using the `memory://` transport) so no external broker is required.

In [ ]:
# Example 1: Construct Exchange / Queue / Connection primitives.
from kombu import Connection, Exchange, Queue

exchange = Exchange('test-exchange', type='direct', durable=False)
queue = Queue('test-queue', exchange=exchange, routing_key='test-key', durable=False)

print(f"exchange.name = {exchange.name}")
print(f"exchange.type = {exchange.type}")
print(f"queue.name = {queue.name}")
print(f"queue.routing_key = {queue.routing_key}")

conn = Connection('memory://')
print(f"conn.transport_cls = {conn.transport_cls}")
print(f"conn.connected (pre-connect) = {conn.connected}")

assert exchange.name == 'test-exchange'
assert queue.exchange is exchange

In [ ]:
# Example 2: Producer publishes a message; Consumer pulls it back (in-memory transport).
from kombu import Connection, Exchange, Queue, Producer, Consumer

exchange = Exchange('demo', type='direct')
queue = Queue('demo-q', exchange=exchange, routing_key='demo')

received = []

def on_message(body, message):
    received.append(body)
    message.ack()

with Connection('memory://') as conn:
    producer = Producer(conn)
    producer.publish({'hello': 'world'}, exchange=exchange, routing_key='demo', declare=[queue])
    with Consumer(conn, [queue], callbacks=[on_message], accept=['json']):
        conn.drain_events(timeout=1)

print(f"received = {received}")
assert received == [{'hello': 'world'}]

In [ ]:
# Example 3: Serializer registry — register, encode, decode round-trip.
from kombu.serialization import registry, dumps, loads

payload = {'a': 1, 'b': [2, 3], 'c': 'hello'}

content_type, content_encoding, data = dumps(payload, serializer='json')
print(f"content_type = {content_type}")
print(f"content_encoding = {content_encoding}")
print(f"encoded bytes = {data[:60]!r}")

decoded = loads(data, content_type, content_encoding)
print(f"decoded = {decoded}")

assert decoded == payload
assert content_type == 'application/json'
assert 'json' in registry._encoders

In [ ]:
# Example 4: SimpleQueue — the high-level put/get API on the memory transport.
from kombu import Connection

with Connection('memory://') as conn:
    sq = conn.SimpleQueue('simple-demo')
    sq.put({'job_id': 42, 'task': 'compute'})
    sq.put('plain-string-payload')

    msg1 = sq.get(block=True, timeout=1)
    print(f"msg1.payload = {msg1.payload}")
    msg1.ack()

    msg2 = sq.get(block=True, timeout=1)
    print(f"msg2.payload = {msg2.payload}")
    msg2.ack()

    sq.close()

assert msg1.payload == {'job_id': 42, 'task': 'compute'}
assert msg2.payload == 'plain-string-payload'

In [ ]:
# Example 5: Routing — direct vs topic exchange behavior, plus dict-form Queue binding.
# Publish with serializer='json': a bare str would go out as text/plain, which the
# Consumer's accept=['json'] allowlist rightly refuses (ContentDisallowed).
from kombu import Connection, Exchange, Queue, Producer, Consumer

topic_ex = Exchange('topic-ex', type='topic')
q_news = Queue('news-q', exchange=topic_ex, routing_key='news.#')
q_sports = Queue('sports-q', exchange=topic_ex, routing_key='sports.#')

received = {'news': [], 'sports': []}

def make_cb(bucket):
    def cb(body, message):
        received[bucket].append(body)
        message.ack()
    return cb

with Connection('memory://') as conn:
    producer = Producer(conn)
    producer.publish('breaking news!', exchange=topic_ex, routing_key='news.world',
                     serializer='json', declare=[q_news, q_sports])
    producer.publish('goal scored', exchange=topic_ex, routing_key='sports.soccer',
                     serializer='json')
    with Consumer(conn, [q_news], callbacks=[make_cb('news')], accept=['json']):
        conn.drain_events(timeout=1)
    with Consumer(conn, [q_sports], callbacks=[make_cb('sports')], accept=['json']):
        conn.drain_events(timeout=1)

print(f"received = {received}")
assert received['news'] == ['breaking news!']
assert received['sports'] == ['goal scored']